<a href="https://colab.research.google.com/github/likhn-1/hybrid-stock-forecasting/blob/main/Preprocessing_and_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preprocessing and Feature Engineering

### Exhibition of Datasets



In [ ]:
# Libraries
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import nltk
import string
import re
from datetime import datetime
from collections import Counter
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize

In [ ]:
# yFinance Apple Historical Data

ticker_symbol = "AAPL"     # ticker symbol

ticker = yf.Ticker(ticker_symbol)   # ticker object

# Fetching historical data from 2016-01-01 to 2024-12-31
historical_data = ticker.history(start="2016-01-01", end="2024-12-31")

# Save historical data to CSV
historical_data[['Open', 'High', 'Low', 'Close', 'Volume']].to_csv('aapl_historical_data.csv')

print(historical_data[['Open', 'High', 'Low', 'Close', 'Volume']])

In [ ]:
# Apple Daily News Dataset

import pandas as pd
news_df = pd.read_csv("/content/apple_news_data.csv")
print(news_df.head())

### Data Preprocessing

To prepare the data for modeling, we performed initial cleaning by removing null values, dropping unused columns (e.g., tags, headline link, and pre-calculated sentiment scores), merge datasets, correcting date formats, and standardizing the text by lowercasing all news content. This ensures consistency and high-quality input for both time-series and NLP-based modeling steps.

In [ ]:
# looking at all features
print("News Data: " + str(news_df.columns.tolist()))
print("YFinance Data: " + str(historical_data.columns.tolist()))

In [ ]:
print(news_df.info())
print('\n')
print(historical_data.info())

In [ ]:
# checking to see null values per column
print("News data: ")
news_df.isna().sum()

In [ ]:
# checking to see null values per column
print("yfinance data: ")
historical_data.isna().sum()

In [ ]:
# dropping sentiment analysis and tags (40% values are null)
news_df = news_df.drop(['sentiment_polarity', 'sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'tags','link'], axis = 1)

In [ ]:
# formatting date to remove the additional timestamp in news_df
news_df['date'] = pd.to_datetime(news_df['date'], utc=True).dt.strftime('%m-%d-%Y')

In [ ]:
# formatting the date column in historical data
historical_data = historical_data.reset_index()
historical_data = historical_data.rename(columns={'index': 'date'})
print(historical_data.columns)
print(historical_data.head())

In [ ]:
historical_data = historical_data.rename(columns={'Date': 'date'})
historical_data = historical_data.drop(columns=['Dividends', 'Stock Splits'])

In [ ]:
news_df['date'] = pd.to_datetime(news_df['date'])
historical_data['date'] = pd.to_datetime(historical_data['date'])

In [ ]:
news_df['date'] = news_df['date'].dt.tz_localize(None)
historical_data['date'] = historical_data['date'].dt.tz_localize(None)

Preprocessing Headlines

In [ ]:
# text preprocessing
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# Lowercase text columns
news_df['title'] = news_df['title'].str.lower()
news_df['content'] = news_df['content'].str.lower()

# Tokenization
news_df['tokenized_title'] = news_df['title'].apply(word_tokenize)
news_df['tokenized_content'] = news_df['content'].apply(word_tokenize)

# Stop word removal
stop_words = set(stopwords.words("english"))
news_df['tokenized_title'] = news_df['tokenized_title'].apply(lambda tokens: [word for word in tokens if word not in stop_words])
news_df['tokenized_content'] = news_df['tokenized_content'].apply(lambda tokens: [word for word in tokens if word not in stop_words])

# Punctuation removal
news_df['tokenized_title'] = news_df['tokenized_title'].apply(lambda tokens: [word for word in tokens if word.isalnum()])
news_df['tokenized_content'] = news_df['tokenized_content'].apply(lambda tokens: [word for word in tokens if word.isalnum()])

# Removing numbers
news_df['tokenized_title'] = news_df['tokenized_title'].apply(lambda tokens: [word for word in tokens if not word.isdigit()])
news_df['tokenized_content'] = news_df['tokenized_content'].apply(lambda tokens: [word for word in tokens if not word.isdigit()])

# Lemmatization (keeping it separate to see the differne between lemmatized and non-lemmatized words)
lemmatizer = WordNetLemmatizer()
news_df['tokenized_title'] = news_df['tokenized_title'].apply(lambda tokens: [lemmatizer.lemmatize(word) for word in tokens])
news_df['tokenized_content'] = news_df['tokenized_content'].apply(lambda tokens: [lemmatizer.lemmatize(word) for word in tokens])

finBERT using HuggingFace and PyTorch for Sentiment Analysis

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline

#creating a pipeline to call finbert
finbert = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone", num_labels = 3)
tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
nlp = pipeline("sentiment-analysis", model = finbert, tokenizer = tokenizer)


In [ ]:
# creating batches of 64 since its ~30k rows
def batch_texts(text_list, batch_size):
  final_list = []
  for i in range(0, len(text_list), batch_size):
      yield text_list[i:i+batch_size]

# texts is created by joining the tokens back into a string
texts = news_df['tokenized_title'].apply(lambda x: ' '.join(x)).tolist()
results = []
for i, batch in enumerate(batch_texts(texts, 64)):
    results.extend(nlp(batch))

    if i % 10 == 0:
        print(f"Processed {i * 64} headlines...")

labels = [res['label'] for res in results]
scores = [res['score'] for res in results]
news_df['sentiment'] = labels
news_df['sentiment_score'] = scores

In [ ]:
news_df.to_csv("news_with_sentiment.csv", index=False)

In [ ]:
news_sentiment_df = pd.read_csv("news_with_sentiment.csv")
news_sentiment_df.head()

Creating Technical Indicators for Historical Data (Feature Engineering)

In [ ]:
# Moving averages
# SMA (Simple Moving Average)
historical_data['SMA_20'] = historical_data['Close'].rolling(window=20).mean()
historical_data['SMA_5'] = historical_data['Close'].rolling(window=5).mean()

# Difference to capture any trend shifts
sma_diff = historical_data['SMA_20'] - historical_data['SMA_5']
historical_data['SMA_diff'] = sma_diff

# EMA (Exponential Moving Average) -  more responsive to recent market changes — useful for short-term trading signals
# Using 12 and 26 for MACD later
historical_data['EMA'] = historical_data['Close'].ewm(span=12, adjust=False).mean()
historical_data['EMA'] = historical_data['Close'].ewm(span=26, adjust=False).mean()

In [ ]:
# RSI (Relative Strength Index)  -  helps identify overbought or oversold conditions
def rsi(prices, period = 14):
  delta = prices.diff()          # calculating teh daily changes
  gain = delta.where(delta > 0,0)
  loss = -delta.where(delta < 0,0)
  avg_gain = gain.rolling(window = period).mean()
  avg_loss = loss.rolling(window = period).mean()
  rs = avg_gain / avg_loss
  rsi = 100.0 - (100.0 / (1.0 + rs))
  return rsi

historical_data['RSI'] = rsi(historical_data['Close'])

In [ ]:
# MACD (Moving Average Convergence Divergence)

macd_line = historical_data['EMA_12'] - historical_data['EMA_26']
signal_line = macd_line.ewm(span=9, adjust=False).mean()
historical_data['MACD'] = macd_line - signal_line

In [ ]:
# lag sentiment - yesterday's headline could make changes in today's stock
news_sentiment_df['sentiment_score_lag'] = news_sentiment_df['sentiment_score'].shift(1)

Merging datasets

In [ ]:
#merge the two datasets
news_sentiment_df['date'] = pd.to_datetime(news_sentiment_df['date'])

merged_df = pd.merge(news_sentiment_df, historical_data, on='date', how='inner')
print(merged_df.shape)

#save csv
merged_df.to_csv('merged_data.csv', index=False)

In [ ]:
print("Merged Data: " + str(merged_df.columns.tolist()))

In [ ]:
# looking at the top 5 rows of news_df
print(merged_df.head(5))